In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
ratings = pd.read_csv("../data/ratings.csv")
movies = pd.read_csv("../data/movies.csv")

In [3]:
user_item = ratings.pivot(
    index="userId",
    columns="movieId",
    values="rating"
)

In [5]:
user_item_filled = user_item.fillna(0)
item_user = user_item_filled.T

Compute item item similarity

In [6]:
item_similarity = cosine_similarity(item_user)

/Users/sabarivishnu/Movierecommendation/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/sabarivishnu/Movierecommendation/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/sabarivishnu/Movierecommendation/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [7]:
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=item_user.index,
    columns=item_user.index
)

In [8]:
item_similarity_df.head()


movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
movieId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.410562,0.296917,0.035573,0.308762,0.376316,0.277491,0.131629,0.232586,0.395573,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.410562,1.000000,0.282438,0.106415,0.287795,0.297009,0.228576,0.172498,0.044835,0.417693,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.296917,0.282438,1.000000,0.092406,0.417802,0.284257,0.402831,0.313434,0.304840,0.242954,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.035573,0.106415,0.092406,1.000000,0.188376,0.089685,0.275035,0.158022,0.000000,0.095598,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.308762,0.287795,0.417802,0.188376,1.000000,0.298969,0.474002,0.283523,0.335058,0.218061,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Find similar movies

In [9]:
def get_similar_movies(movie_id, k=5):
    similarities = item_similarity_df[movie_id]
    similarities = similarities.drop(movie_id)
    return similarities.sort_values(ascending=False).head(k)
get_similar_movies(movie_id=1, k=5)

movieId
3114    0.572601
480     0.565637
780     0.564262
260     0.557388
356     0.547096
Name: 1, dtype: float64

Recommend movies for a user

In [13]:
def recommend_movies_item_based(target_user_id, n_recommendations=5):
    
    user_ratings = user_item.loc[target_user_id].dropna()

    scores = pd.Series(dtype=float)

    for movie_id, rating in user_ratings.items():
        similar_movies = get_similar_movies(movie_id, k=10)
        scores = scores.add(similar_movies * rating, fill_value=0)

    # FIX HERE
    scores = scores.drop(user_ratings.index, errors="ignore")

    return scores.sort_values(ascending=False).head(n_recommendations)


In [14]:
recommendations = recommend_movies_item_based(1, 5)
recommendations

movieId
2918    44.501502
1200    31.087732
380     27.243123
2087    26.267612
589     24.723248
dtype: float64

In [15]:
movies[movies["movieId"].isin(recommendations.index)][["movieId", "title"]]

,movieId,title
337,380,True Lies (1994)
507,589,Terminator 2: Judgment Day (1991)
902,1200,Aliens (1986)
1550,2087,Peter Pan (1953)
2195,2918,Ferris Bueller's Day Off (1986)


In [16]:
def predict_item_cf(user_id, movie_id):
    
    if movie_id not in item_similarity_df.columns:
        return np.nan
    
    user_ratings = user_item.loc[user_id].dropna()
    
    similarities = item_similarity_df[movie_id].dropna()

    numerator = 0
    denominator = 0

    for m, rating in user_ratings.items():
        if m in similarities:
            sim = similarities[m]
            numerator += sim * rating
            denominator += abs(sim)

    if denominator == 0:
        return np.nan

    return numerator / denominator


In [17]:
def predict_item_cf(user_id, movie_id):
    
    if movie_id not in item_similarity_df.columns:
        return np.nan
    
    user_ratings = user_item.loc[user_id].dropna()
    
    similarities = item_similarity_df[movie_id].dropna()

    numerator = 0
    denominator = 0

    for m, rating in user_ratings.items():
        if m in similarities:
            sim = similarities[m]
            numerator += sim * rating
            denominator += abs(sim)

    if denominator == 0:
        return np.nan

    return numerator / denominator


In [19]:
from sklearn.metrics import mean_squared_error
actual = []
predicted = []

for row in ratings.sample(500).itertuples():
    pred = predict_item_cf(row.userId, row.movieId)

    if not np.isnan(pred):
        actual.append(row.rating)
        predicted.append(pred)

rmse_item = np.sqrt(mean_squared_error(actual, predicted))
rmse_item


np.float64(0.9050958517904553)

In [24]:
import json

with open("../outputs/rmse_results.json") as f:
    results = json.load(f)

rmse_user = results["rmse_user"]
results["rmse_item"] = float(rmse_item)

with open("../outputs/rmse_results.json", "w") as f:
    json.dump(results, f)
